# CFTC TFF 日期格式机械修复

沿用同一2011原件、精确纳指Mini代码209742；接受原件ISO报告日期，身份通过后仅执行一次剩余2026请求。

In [1]:
import csv,io,json,hashlib,subprocess,tarfile,tempfile,shlex,zipfile,re
from pathlib import Path
from datetime import datetime,timezone
from src.data.nas_market_data import _settings,nas_api_call
root=Path.cwd();card=root/'research/stage1_cftc_nq_tff_source_probe_20261023';plan=json.loads((card/'plan.json').read_text());repair=json.loads((card/'mechanical_parser_identity_repair_plan.json').read_text());repair2=json.loads((card/'mechanical_report_date_repair_plan.json').read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
raw=root/'data/snapshots/stage1_cftc_nq_tff_source_probe_20261023';assert repair['frozen'] and repair['original_plan_sha256']==sha(card/'plan.json') and repair['original_attempt1_result_sha256']==sha(card/'source_attempt1_parser_failure.json') and repair2['frozen'] and repair2['first_repair_plan_sha256']==sha(card/'mechanical_parser_identity_repair_plan.json')
first=raw/'fut_fin_txt_2011.zip';assert sha(first)==repair['saved_2011_raw_sha256'] and sha(raw/'fut_fin_txt_2011.zip.headers')==repair['saved_2011_header_sha256'] and not (raw/'fut_fin_txt_2026.zip').exists() and not (card/'repair2_run_state.json').exists()
state={'status':'started','cumulative_GET_attempts':1,'repair_GET_attempts':0,'model_fits':0,'native_accounts':0};(card/'repair2_run_state.json').write_text(json.dumps(state,indent=2)+'\n')
def parse(year,body):
 with zipfile.ZipFile(body) as z:
  members=[x for x in z.namelist() if x.lower().endswith(('.txt','.csv')) and not x.startswith('__MACOSX/')];assert len(members)==1
  content=z.read(members[0])
 try:decoded=content.decode('utf-8-sig')
 except UnicodeDecodeError:decoded=content.decode('cp1252')
 table=csv.DictReader(io.StringIO(decoded));columns=table.fieldnames;rows=list(table);assert columns and len(rows)>200
 normalized={re.sub(r'[^a-z0-9]','',x.lower()):x for x in columns};needed=['marketandexchangenames','reportdateasm mddyyyy'.replace(' ',''),'cftccontractmarketcode','openinterestall','levmoneypositionslongall','levmoneypositionsshortall']
 assert all(x in normalized for x in needed),(needed,sorted(normalized)[:30])
 market,report,code,oi,long,short=[normalized[x] for x in needed]
 nq=[row for row in rows if row[code].strip()=='209742' and 'NASDAQ' in row[market].upper() and 'MINI' in row[market].upper() and 'CHICAGO MERCANTILE' in row[market].upper()];assert len(nq)>=20

 def parse_day(text):
  text=text.strip()
  try:return datetime.strptime(text,'%Y-%m-%d').date()
  except ValueError:return datetime.strptime(text,'%m/%d/%Y').date()
 dates=[parse_day(row[report]) for row in nq];assert len(dates)==len(set(dates)) and min(dates).year==year and max(dates).year==year
 number=lambda x:float(x.replace(',','').strip())
 assert all(number(row[oi])>0 and number(row[long])>=0 and number(row[short])>=0 for row in nq)
 return {'year':year,'status':'official_TFF_Emini_NQ_209742_identity_passed','raw_sha256':sha(body),'zip_member':members[0],'zip_member_bytes':len(content),'total_report_rows':len(rows),'NQ_rows':len(nq),'NQ_first_report_date':str(min(dates)),'NQ_last_report_date':str(max(dates)),'NQ_market_names':sorted({row[market].strip() for row in nq}),'NQ_contract_code':'209742','Tuesday_report_date_share':sum(d.weekday()==1 for d in dates)/len(dates),'required_column_names':{'market':market,'report_date':report,'contract_code':code,'open_interest':oi,'leveraged_money_long':long,'leveraged_money_short':short},'report_date_is_not_release_date':True}
checks=[parse(2011,first)];receipts=json.loads((card/'capture_receipts.json').read_text());assert len(receipts)==1 and receipts[0]['year']==2011
spec=plan['exact_requests'][1];r=plan['request'];body=raw/'fut_fin_txt_2026.zip';headers=raw/'fut_fin_txt_2026.zip.headers';started=datetime.now(timezone.utc).isoformat();state['repair_GET_attempts']=1;state['cumulative_GET_attempts']=2;(card/'repair2_run_state.json').write_text(json.dumps(state,indent=2)+'\n')
cmd=['curl','--http1.1','--noproxy','*','--connect-timeout',str(r['connect_timeout_seconds']),'--max-time',str(r['total_timeout_seconds_each']),'--max-filesize',str(r['max_response_bytes_each']),'--fail-with-body','-sS','-L','-D',str(headers),'-o',str(body),'-A',r['user_agent'],'-w','%{http_code} %{url_effective}',spec['url']]
try:
 run=subprocess.run(cmd,capture_output=True,text=True,timeout=110);parts=run.stdout.rsplit(' ',1);status=int(parts[0]) if len(parts)==2 and parts[0].isdigit() else None
 receipt={'year':2026,'url':spec['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'curl_exit_code':run.returncode,'http_status':status,'final_url':parts[-1] if parts else None,'stderr':run.stderr[:350]}
except Exception as exc:receipt={'year':2026,'url':spec['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'transport_exception':type(exc).__name__+':'+str(exc)[:180]}
receipt.update(raw_sha256=sha(body) if body.exists() else None,raw_bytes=body.stat().st_size if body.exists() else 0,headers_sha256=sha(headers) if headers.exists() else None,headers_bytes=headers.stat().st_size if headers.exists() else 0);receipts.append(receipt)
failure=None
if receipt.get('curl_exit_code')!=0 or receipt.get('http_status')!=200 or not body.exists():failure='2026_transport_failure'
else:
 try:checks.append(parse(2026,body))
 except Exception as exc:failure='2026_identity_failure';checks.append({'year':2026,'status':failure,'error':type(exc).__name__+':'+str(exc)[:240],'raw_sha256':receipt['raw_sha256']})
(card/'repair_capture_receipts.json').write_text(json.dumps(receipts,indent=2)+'\n');manifest={'repair_plan_sha256':sha(card/'mechanical_parser_identity_repair_plan.json'),'repair2_plan_sha256':sha(card/'mechanical_report_date_repair_plan.json'),'files':[{'name':p.name,'sha256':sha(p),'bytes':p.stat().st_size} for p in sorted(raw.iterdir()) if p.is_file()],'repair_capture_receipts_sha256':sha(card/'repair_capture_receipts.json')};(card/'repair_source_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='cftc-tff-repair-') as td:
 archive=Path(td)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as t:
  for x in manifest['files']:t.add(raw/x['name'],arcname='raw/'+x['name'])
  for name in ('repair_source_manifest.json','repair_capture_receipts.json','source_attempt1_parser_failure.json','mechanical_parser_identity_repair_plan.json','mechanical_report_date_repair_plan.json'):t.add(card/name,arcname=name)
 digest=sha(archive);evidence=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact'];assert evidence['sha256']==digest and evidence['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 with tempfile.TemporaryDirectory(prefix='cftc-tff-repair-restore-') as rd:
  restored=Path(rd)/'restore.tar.gz'
  with restored.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,timeout=120)
  assert run.returncode==0 and sha(restored)==digest
  with tarfile.open(restored,'r:gz') as t:
   for x in manifest['files']:
    member=t.extractfile('raw/'+x['name']);assert member and hashlib.sha256(member.read()).hexdigest()==x['sha256']
result={'status':'official_CFTC_TFF_Emini_NQ_2011_2026_identity_passed_release_clock_pending' if not failure and len(checks)==2 else 'bounded_CFTC_TFF_Emini_NQ_repair_partial_or_failed','failure':failure,'cumulative_GET_attempts':2,'repair_GET_attempts':1,'checks':checks,'initial_partial_NAS_archive_sha256':json.loads((card/'source_attempt1_parser_failure.json').read_text())['NAS_archive_sha256'],'repair_NAS_archive_sha256':digest,'NAS_remote_and_scoped_restore_verified':True,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,'historical_release_clock_qualified':False};(card/'repair_result.json').write_text(json.dumps(result,indent=2)+'\n');state['status']='completed' if not failure else 'failed';(card/'repair2_run_state.json').write_text(json.dumps(state,indent=2)+'\n');print({'status':result['status'],'failure':failure,'checks':checks,'NAS_sha256':digest})


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'status': 'bounded_CFTC_TFF_Emini_NQ_repair_partial_or_failed', 'failure': '2026_identity_failure', 'checks': [{'year': 2011, 'status': 'official_TFF_Emini_NQ_209742_identity_passed', 'raw_sha256': '583a93474847f1070b213e2b12cfa73c48d9ba05f24f7d001a338de125468b57', 'zip_member': 'FinFutYY.txt', 'zip_member_bytes': 1502514, 'total_report_rows': 1879, 'NQ_rows': 52, 'NQ_first_report_date': '2011-01-04', 'NQ_last_report_date': '2011-12-27', 'NQ_market_names': ['NASDAQ-100 STOCK INDEX (MINI) - CHICAGO MERCANTILE EXCHANGE'], 'NQ_contract_code': '209742', 'Tuesday_report_date_share': 1.0, 'required_column_names': {'market': 'Market_and_Exchange_Names', 'report_date': 'Report_Date_as_MM_DD_YYYY', 'contract_code': 'CFTC_Contract_Market_Code', 'open_interest': 'Open_Interest_All', 'leveraged_money_long': 'Lev_Money_Positions_Long_All', 'leveraged_money_short': 'Lev_Money_Positions_Short_All'}, 'report_date_is_not_release_date': True}, {'year': 2026, 'status': '2026_identity_failure', 'error': 

## 中文结论

2011日期值解析成功，第二次且最后一次冻结 GET取得2026原件并完成 NAS 备份；2026日期列名变化使本次解析停止。

完整科学决定见 `conclusion.md`。